# LC 155 — Min Stack
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Stack Design
**Pattern:** Parallel Min-Tracking Stack

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Run a second stack in
parallel. At every push, record the minimum so far
on the min-stack. Pop both together. The top of the
min-stack is always the current minimum — O(1).
</div>

## Official Problem Statement

Design a stack that supports push, pop, top, and
retrieving the minimum element in constant time.

Implement the `MinStack` class:
- `MinStack()` initializes the stack object.
- `void push(int val)` pushes the element `val`
  onto the stack.
- `void pop()` removes the element on the top.
- `int top()` gets the top element.
- `int getMin()` retrieves the minimum element.

You must implement a solution with `O(1)` time
complexity for each function.

**Example 1:**
```
Input:
["MinStack","push","push","push","getMin",
 "pop","top","getMin"]
[[],[-2],[0],[-3],[],[],[],[]]
Output: [null,null,null,null,-3,null,0,-2]
```

**Constraints:**
- `-2^31 <= val <= 2^31 - 1`
- Methods `pop`, `top`, `getMin` will always be
  called on non-empty stacks
- At most `3 * 10^4` calls will be made

## What This Is Actually Asking

Build a stack that also knows, at any moment, what
the smallest value currently in the stack is.
The tricky part: when you pop the current minimum
off, the stack must still know the new minimum
instantly — without scanning.
Every operation must be O(1): no scanning allowed.

## Walk Through an Example by Hand

```
operations: push(-2), push(0), push(-3),
            getMin, pop, top, getMin

stack    min_stack    action
[]       []           start
[-2]     [-2]         push(-2)  min=min(-2, n/a)=-2
[-2,0]   [-2,-2]      push(0)   min=min(0,-2)=-2
[-2,0,-3][-2,-2,-3]   push(-3)  min=min(-3,-2)=-3

getMin -> min_stack[-1] = -3  ✓

pop    -> pop both stacks
[-2,0]   [-2,-2]

top    -> stack[-1] = 0  ✓

getMin -> min_stack[-1] = -2  ✓
(previous minimum is now exposed — no scan needed)
```

## The Picture

```
Two stacks run side by side:

  main stack    min stack
  ----------    ---------
  push(-2):  [-2]        [-2]      min so far = -2
  push( 0):  [-2, 0]     [-2, -2]  min so far = -2
  push(-3):  [-2, 0, -3] [-2,-2,-3] min so far=-3

  getMin()  -> min_stack top = -3  O(1) ✓

  pop()     -> pop BOTH
             [-2, 0]     [-2, -2]

  getMin()  -> min_stack top = -2  O(1) ✓

Key: min_stack[i] = minimum of main_stack[0..i]
     They stay perfectly in sync.
     Popping -3 from main automatically reveals
     the previous minimum on min_stack.
```

## When To Use This Pattern

- When a data structure needs O(1) access to a
  running aggregate (min, max), think **parallel
  tracking stack**
- When popping the current min/max must instantly
  reveal the previous one, think **shadow stack that
  mirrors the main one**
- When extending to max-stack, think **same pattern —
  max_stack[-1] = max of all elements so far**
- When space is a concern, think **only push to
  min_stack when a new minimum is found** (space-
  optimised variant with tuples)

## The Approach

Maintain two lists: the main stack for values and a
min-stack that stores the running minimum at each
depth.
On push, append the new value to the main stack and
append the smaller of the new value and the current
min-stack top to the min-stack.
On pop, remove the top from both stacks together.
getMin always returns the top of the min-stack.

In [1]:
# No imports needed — plain Python lists as stacks

In [2]:
def test_harness(cls):
    """
    Tests MinStack by replaying operation sequences.
    Each test is (ops_list, args_list, expected_list).
    None in expected means the op returns nothing.
    """
    tests = [
        # ops, args, expected outputs (None = void return)
        (
            ["push","push","push","getMin","pop",
             "top","getMin"],
            [[-2],[0],[-3],[],[],[],[]],
            [None,None,None,-3,None,0,-2]
        ),
        (
            ["push","push","getMin","pop","getMin"],
            [[5],[3],[],[],[]],
            [None,None,3,None,5]
        ),
        (
            ["push","getMin","push","getMin"],
            [[1],[],[2],[]],
            [None,1,None,1]
        ),
        (
            ["push","push","push","top","getMin"],
            [[2],[1],[3],[],[]],
            [None,None,None,3,1]
        ),
    ]

    passed = 0
    for i, (ops, args, expected) in enumerate(tests):
        ms = cls()
        results = []
        for op, arg in zip(ops, args):
            if op == "push":
                ms.push(arg[0]); results.append(None)
            elif op == "pop":
                ms.pop(); results.append(None)
            elif op == "top":
                results.append(ms.top())
            elif op == "getMin":
                results.append(ms.getMin())

        ok = results == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | ops={ops} | "
            f"expected={expected} | got={results}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [6]:
class MinStack:
    """
    Stack with O(1) push, pop, top, and getMin.

    Maintain two lists: stack (values) and min_stack
    (running minimum at each depth). On push, append
    to both — min_stack gets min(val, current_min).
    On pop, remove top from both. getMin returns
    min_stack[-1].

    Time:  O(1) — all four operations
    Space: O(n) — two stacks, each at most n elements
    """

    def __init__(self):
        self.stack     = []    # main stack
        self.min_stack = []    # tracks current min at every level

    def push(self, val: int) -> None:
        self.stack.append(val)
        cur_min = val if not self.min_stack else min (val ,self.min_stack[-1])
        self.min_stack.append(cur_min)

    def pop(self) -> None:
        self.stack.pop()
        self.min_stack.pop()
        
    def top(self) -> int:
        return self.stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]


# Quick debug — run this cell while building
ms = MinStack()
ms.push(-2)
ms.push(0)
ms.push(-3)
print(ms.getMin())  # -3
ms.pop()
print(ms.top())     # 0
print(ms.getMin())  # -2
test_harness(MinStack)

-3
0
-2
Test 1: PASSED | ops=['push', 'push', 'push', 'getMin', 'pop', 'top', 'getMin'] | expected=[None, None, None, -3, None, 0, -2] | got=[None, None, None, -3, None, 0, -2]
Test 2: PASSED | ops=['push', 'push', 'getMin', 'pop', 'getMin'] | expected=[None, None, 3, None, 5] | got=[None, None, 3, None, 5]
Test 3: PASSED | ops=['push', 'getMin', 'push', 'getMin'] | expected=[None, 1, None, 1] | got=[None, 1, None, 1]
Test 4: PASSED | ops=['push', 'push', 'push', 'top', 'getMin'] | expected=[None, None, None, 3, 1] | got=[None, None, None, 3, 1]

4/4 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(MinStack)

## Complexity

| Approach | Time (getMin) | Space |
|---|---|---|
| Scan whole stack on getMin | O(n) | O(1) extra |
| Parallel min-stack | O(1) | O(n) extra |

The parallel stack trades O(n) extra space to make
every operation O(1) — the standard interview
answer for this problem.

## Real World Connection

At Citi, the telemetry alert system must always
know the current minimum CPU reading across the
active monitoring window in O(1) — scanning 6,000
servers on every alert check is too slow.
A min-stack running alongside the ingest buffer
provides the floor value instantly as readings are
pushed and expired.
The same pattern applies in the ETL pipeline when
tracking the watermark (earliest unprocessed
timestamp) as events are consumed from Kafka —
the min-stack gives the watermark in O(1) without
rescanning the entire partition buffer.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra